# Hurdle forecasts of bilateral migration flows

One-window-ahead forecasts of the Abel & Cohen bilateral flows. Existence: BART probit. Volume: zero-truncated
negative binomial in Stan (`stan/hurdle_volume.stan`). Input: `outputs/data_prep/panel_final.csv`.
Outputs: `outputs/hurdle/<RUN>/` (metrics, forecasts, Stan and BART draws).

$Y_{ijt}$: migrants from origin $i$ to destination $j$ between mid-$t$ and mid-$(t+5)$, $t\in\{1990, 1995, \dots, 2015\}$;
$t-5$ is the previous window; $k$ is the M49 sub-region of the origin.

```bash
pip install "stochtree==0.4.5" xgboost cmdstanpy
python -c "import cmdstanpy; cmdstanpy.install_cmdstan(version='2.40.0', cores=8)"
jupyter nbconvert --to script hurdle_forecast.ipynb && python -u hurdle_forecast.py 2>&1 | tee run.log
```

In [ ]:
import json
import logging
import os
import shutil
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from cmdstanpy import CmdStanModel
from cmdstanpy.utils import get_logger
from IPython.display import display
from joblib import Parallel, cpu_count, delayed
from scipy import stats
from scipy.special import logsumexp, ndtr
from sklearn.metrics import roc_auc_score

if '__file__' in globals():               # run as a script: paths are relative to this folder
    os.chdir(Path(__file__).resolve().parent)
warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)
warnings.filterwarnings('ignore', message='IProgress not found')
get_logger().setLevel(logging.ERROR)
pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 40)

T = 2015                                  # forecast window: mid-2015 to mid-2020
CAL_YEARS = [T - 10, T - 5]               # windows used to calibrate the thresholds
COUNTRIES = None                          # None: all countries; or a list of ISO3 codes
RUN = 'main'                              # output folder: give each model version its own name
RUNS = Path('../outputs/hurdle')
OUT = RUNS / RUN

N_CHAINS, ITER_WARMUP, ITER_SAMPLING = 4, 1000, 1200                    # Stan
MAX_TREEDEPTH, ADAPT_DELTA, SEED = 12, 0.95, 42
NTREE, K_BART, POWER, BASE = 200, 2.0, 2.0, 0.95                         # BART prior
BART_CHAINS, BART_GFR, BART_BURN, BART_DRAWS, SEED_BART = 16, 10, 100, 300, 42   # 16 x 300 = 4,800 Stan draws
LAMBDA = 1.0                              # weight of the relative error in the loss

OUT.mkdir(parents=True, exist_ok=True)
np.random.seed(SEED)
print(f"Run {RUN}: forecast of {T} | Stan {N_CHAINS} x ({ITER_WARMUP} + {ITER_SAMPLING}) | "
      f"BART {BART_CHAINS} x ({BART_GFR} XBART sweeps + {BART_BURN} + {BART_DRAWS})")

## Functions

In [ ]:
def _ess(z):
    """ESS of each column of z (chains, draws, columns), with Geyer's initial monotone sequence; also R-hat."""
    k, h, _ = z.shape
    W = z.var(1, ddof=1).mean(0)
    var = (h - 1) / h * W + z.mean(1).var(0, ddof=1)
    zc = z - z.mean(1, keepdims=True)
    fz = np.fft.rfft(zc, n=2 * h, axis=1)
    acov = np.fft.irfft(fz * np.conj(fz), n=2 * h, axis=1)[:, :h] / h
    with np.errstate(divide='ignore', invalid='ignore'):
        rho = 1 - (W - acov.mean(0)) / var
        rho[0] = 1.0
        L = h // 2
        pair = rho[0:2 * L:2] + rho[1:2 * L:2]
        keep = np.cumprod(pair > 0, axis=0).astype(bool)
        pair = np.minimum.accumulate(np.where(keep, pair, np.inf), axis=0)
        tau = np.maximum(-1 + 2 * np.where(keep, pair, 0).sum(0), 1 / np.log10(k * h))
        ess = np.minimum(k * h / tau, k * h * np.log10(k * h))
        return np.nan_to_num(ess, nan=0.0), np.sqrt(var / W)


def diagnostics(x):
    """Rank-normalised split R-hat, bulk ESS and tail ESS (Vehtari et al. 2021) per column of x (chains, draws, columns)."""
    h = x.shape[1] // 2
    x = np.concatenate([x[:, :h], x[:, h:2 * h]])                      # split chains
    k, m = x.shape[0], x.shape[2]
    r = stats.rankdata(x.reshape(k * h, m), axis=0).reshape(k, h, m)
    ess_bulk, rhat = _ess(stats.norm.ppf((r - 0.375) / (k * h + 0.25)))
    q05, q95 = np.quantile(x.reshape(k * h, m), [0.05, 0.95], axis=0)
    ess_tail = np.minimum(_ess((x <= q05).astype(float))[0], _ess((x <= q95).astype(float))[0])
    return rhat, ess_bulk, ess_tail


def ztnb_draw(eta, phi, rng):
    """Draws of Y | Y > 0 for Y ~ NB2(mean exp(eta), dispersion phi), elementwise."""
    mu, phi = np.exp(np.clip(eta, -50, 50)), np.clip(phi, 1e-8, 1e8)
    p = phi / (phi + mu)
    p0 = np.exp(-phi * np.log1p(mu / phi))                              # P(Y = 0)
    v = np.ones(mu.shape, dtype=np.int64)
    todo = p0 < 0.5                                                      # rejection: fewer than 2 tries on average
    while todo.any():
        v[todo] = rng.negative_binomial(phi[todo], p[todo])
        todo &= v == 0
    inv = (p0 >= 0.5) & (1 - p0 > 1e-12)                                 # inverse CDF on (p0, 1); else Y = 1
    u = np.minimum(p0[inv] + (1 - p0[inv]) * rng.random(inv.sum()), np.nextafter(1.0, 0))
    v[inv] = np.maximum(stats.nbinom.ppf(u, phi[inv], p[inv]), 1)
    return v


def wmedian(v, w):
    """Weighted median of each column."""
    o = np.argsort(v, axis=0)
    vs, cw = np.take_along_axis(v, o, 0), np.cumsum(np.take_along_axis(w, o, 0), 0)
    return vs[(cw < 0.5 * cw[-1]).sum(0), np.arange(v.shape[1])]


def bayes_action(v, pb, w):
    """Weighted median of the hurdle predictive (atom 1 - p at 0, p * ZTNB above), weights w(y) = 1 + 100 lambda / (y + 1)."""
    w0 = (1 - pb).mean(0) * (1 + 100 * LAMBDA)
    o = np.argsort(v, axis=0)
    vs = np.take_along_axis(v, o, 0)
    cw = w0 + np.cumsum(np.take_along_axis(pb * w, o, 0), 0) / len(v)
    k = np.minimum((cw < 0.5 * cw[-1]).sum(0), len(v) - 1)
    return np.where(w0 >= 0.5 * cw[-1], 0, vs[k, np.arange(v.shape[1])])


def crps(x, y):
    """CRPS from draws: E|X - y| - E|X - X'| / 2."""
    xs, n = np.sort(x, axis=0), len(x)
    return np.abs(x - y).mean(0) - ((2 * np.arange(1, n + 1) - n - 1)[:, None] * xs).sum(0) / n ** 2


def ztnb_logpmf(y, eta, phi):
    """log of the posterior mean of the ZTNB pmf at y > 0."""
    lp = stats.nbinom.logpmf(y, phi, phi / (phi + np.exp(eta))) - np.log(-np.expm1(-phi * np.log1p(np.exp(eta) / phi)))
    return logsumexp(lp, axis=0) - np.log(len(eta))


def thresholds(p, f, v, c, lam):
    """Threshold per M49 sub-region minimising the sum of |f - forecast| (1 + 100 lam / (f + 1)) on the calibration rows."""
    grid = np.quantile(p, np.linspace(0.05, 0.9995, 250))
    thr = {}
    for k in np.unique(c):
        m = c == k
        err = [np.abs(f[m] - np.where(p[m] >= t, v[m], 0.0)) for t in grid]
        loss = [e.sum() + lam * 100 * (e / (f[m] + 1)).sum() for e in err]
        thr[k] = float(grid[int(np.argmin(loss))])   # first minimum
    return thr


def read_stan_csv(files, names, draws):
    """Draws of the named variables in CmdStan CSV files: array (chains x draws, columns), chains stacked and columns
    in file order, with the column names. A name is a variable ('eps') or one element ('eps.17')."""
    out = None
    for c, f in enumerate(files):
        with open(f) as fh:
            header = next(line for line in fh if not line.startswith('#')).rstrip('\n').split(',')
            ix = np.array([i for i, h in enumerate(header) if h in names or h.split('.')[0] in names], dtype=int)
            if out is None:
                out = np.empty((len(files) * draws, len(ix)))
            s = c * draws
            for line in fh:
                if not line.startswith('#'):
                    row = np.fromstring(line, sep=',')
                    assert len(row) == len(header), f"{f}: draw {s - c * draws + 1} unreadable"
                    out[s] = row[ix]
                    s += 1
        assert s == (c + 1) * draws, f"{f}: {s - c * draws} draws, {draws} expected"
    return out, [header[i] for i in ix]


def bart_agreement(z, y, n_sub=2000):
    """Existence scores of p (mean of Phi over the draws z: chains, draws, rows) and how far the chains are apart.
    se of p: Monte Carlo standard error of p, from the spread of the chain means."""
    pc = np.stack([ndtr(zc).mean(0, dtype=float) for zc in z])           # p of each chain
    p = np.clip(pc.mean(0), 1e-9, 1 - 1e-9)
    lc = np.clip(pc, 1e-9, 1 - 1e-9)
    ls_c = np.where(y, np.log(lc), np.log1p(-lc)).mean(1)
    se = pc.std(0, ddof=1) / np.sqrt(len(z))
    sub = np.random.default_rng(0).choice(z.shape[2], min(n_sub, z.shape[2]), replace=False)
    rh, eb, _ = diagnostics(z[:, :, sub].astype(float))
    return {'log score': np.where(y, np.log(p), np.log1p(-p)).mean(), 'Brier': np.mean((p - y) ** 2),
            'AUC': roc_auc_score(y, p), 'log score, worst chain': ls_c.min(), 'log score, best chain': ls_c.max(),
            'mean |p chain - p|': np.abs(pc - pc.mean(0)).mean(), 'se of p, median': np.median(se),
            'se of p, 99th percentile': np.percentile(se, 99), 'R-hat, median': np.median(rh),
            'bulk ESS, median': np.median(eb)}


def saved_bart_draws(folder):
    """BART draws saved by a finished run in folder/bart: zb, N_h, covariate split shares, fitting time."""
    info = json.loads((folder / 'bart' / 'info.json').read_text())
    return np.load(folder / 'bart' / 'zb.npy'), info['N_h'], pd.Series(info['importance']), info['t_bart']


def saved_stan_draws(folder):
    """CSV files of the Stan run saved in folder/stan (the latest one) and its sampling time in seconds."""
    files = sorted((folder / 'stan').glob('hurdle_volume-*.csv'))
    stamp = max(f.name.rsplit('_', 1)[0] for f in files)
    csv = [str(f) for f in files if f.name.rsplit('_', 1)[0] == stamp]
    assert len(csv) == N_CHAINS, f"{len(csv)} CSV files for {stamp}, {N_CHAINS} expected"
    t = max((float(line.split()[0]) for f in (folder / 'stan').glob(stamp + '_*-stdout.txt')
             for line in open(f) if line.strip().endswith('seconds (Total)')), default=float('nan'))
    return csv, t


rng_t = np.random.default_rng(1)   # check of the sampler: E[Y | Y > 0] = mu / (1 - p0)
for mu_, phi_ in [(0.05, 0.5), (0.8, 2.0), (3.0, 0.3), (50.0, 1.5), (2000.0, 0.8)]:
    v_ = ztnb_draw(np.full(200_000, np.log(mu_)), np.full(200_000, phi_), rng_t)
    m_ = mu_ / (1 - (phi_ / (phi_ + mu_)) ** phi_)
    assert v_.min() >= 1 and abs(v_.mean() - m_) < 4 * v_.std() / np.sqrt(len(v_)), (mu_, phi_, v_.mean(), m_)
print("ZTNB sampler: E[Y | Y > 0] = mu / (1 - p0) on 5 cases")

## Data and covariates

Network features of window $t$, from the corridors open in window $t-5$:
$\text{transitivity}_{ijt}=\deg^{\text{out}}_{i}\cdot\deg^{\text{in}}_{j}$ and
$\text{A2}_{ijt}=\log\big(1+\#\{k:\ i\to k \text{ and } k\to j \text{ open}\}\big)$.

Stocks (UN DESA 2020): $S_{i\to j,t}$ = people born in $i$ living in $j$ at mid-$t$. In $x_{ijt}$:
$\log(1+S_{i\to j,t})$ (diaspora) and $\log(1+S_{j\to i,t})$ (return).

In [ ]:
ISO3_TO_M49 = {
    'DNK': 11, 'EST': 11, 'FIN': 11, 'ISL': 11, 'IRL': 11, 'LVA': 11, 'LTU': 11, 'NOR': 11, 'SWE': 11, 'GBR': 11,
    'ALB': 12, 'AND': 12, 'BIH': 12, 'HRV': 12, 'GRC': 12, 'ITA': 12, 'MLT': 12, 'MNE': 12, 'MKD': 12, 'PRT': 12, 'SRB': 12, 'SVN': 12, 'ESP': 12,
    'AUT': 13, 'BEL': 13, 'FRA': 13, 'DEU': 13, 'LIE': 13, 'LUX': 13, 'MCO': 13, 'NLD': 13, 'CHE': 13,
    'BLR': 14, 'BGR': 14, 'CZE': 14, 'HUN': 14, 'POL': 14, 'MDA': 14, 'ROU': 14, 'RUS': 14, 'SVK': 14, 'UKR': 14,
    'DZA': 15, 'EGY': 15, 'LBY': 15, 'MAR': 15, 'SDN': 15, 'TUN': 15, 'ESH': 15,
    'BEN': 16, 'BFA': 16, 'CPV': 16, 'CIV': 16, 'GMB': 16, 'GHA': 16, 'GIN': 16, 'GNB': 16, 'LBR': 16, 'MLI': 16, 'MRT': 16, 'NER': 16, 'NGA': 16, 'SEN': 16, 'SLE': 16, 'TGO': 16,
    'BDI': 17, 'COM': 17, 'DJI': 17, 'ERI': 17, 'ETH': 17, 'KEN': 17, 'MDG': 17, 'MWI': 17, 'MUS': 17, 'MOZ': 17, 'REU': 17, 'RWA': 17, 'SYC': 17, 'SOM': 17, 'SSD': 17, 'TZA': 17, 'UGA': 17, 'ZMB': 17, 'ZWE': 17,
    'AGO': 18, 'CMR': 18, 'CAF': 18, 'TCD': 18, 'COD': 18, 'COG': 18, 'GNQ': 18, 'GAB': 18, 'STP': 18,
    'BWA': 19, 'LSO': 19, 'NAM': 19, 'ZAF': 19, 'SWZ': 19,
    'CAN': 21, 'MEX': 21, 'USA': 21,
    'BLZ': 22, 'CRI': 22, 'SLV': 22, 'GTM': 22, 'HND': 22, 'NIC': 22, 'PAN': 22,
    'ATG': 23, 'BHS': 23, 'BRB': 23, 'CUB': 23, 'DMA': 23, 'DOM': 23, 'GLP': 23, 'GRD': 23, 'HTI': 23, 'JAM': 23, 'KNA': 23, 'LCA': 23, 'MTQ': 23, 'VCT': 23, 'TTO': 23, 'ABW': 23, 'PRI': 23,
    'ARG': 24, 'BOL': 24, 'BRA': 24, 'CHL': 24, 'COL': 24, 'ECU': 24, 'GUF': 24, 'GUY': 24, 'PRY': 24, 'PER': 24, 'SUR': 24, 'URY': 24, 'VEN': 24,
    'CHN': 30, 'HKG': 30, 'JPN': 30, 'KOR': 30, 'MAC': 30, 'MNG': 30, 'PRK': 30,
    'AFG': 34, 'BGD': 34, 'BTN': 34, 'IND': 34, 'IRN': 34, 'MDV': 34, 'NPL': 34, 'PAK': 34, 'LKA': 34,
    'BRN': 35, 'KHM': 35, 'IDN': 35, 'LAO': 35, 'MYS': 35, 'MMR': 35, 'PHL': 35, 'SGP': 35, 'THA': 35, 'TLS': 35, 'VNM': 35,
    'ARM': 145, 'AZE': 145, 'BHR': 145, 'CYP': 145, 'GEO': 145, 'IRQ': 145, 'ISR': 145, 'JOR': 145, 'KWT': 145, 'LBN': 145, 'OMN': 145, 'QAT': 145, 'SAU': 145, 'PSE': 145, 'SYR': 145, 'TUR': 145, 'ARE': 145, 'YEM': 145,
    'KAZ': 143, 'KGZ': 143, 'TJK': 143, 'TKM': 143, 'UZB': 143,
    'AUS': 53, 'FJI': 53, 'NZL': 53, 'PNG': 53, 'SLB': 53, 'VUT': 53, 'WSM': 53, 'TON': 53, 'KIR': 53, 'FSM': 53, 'GUM': 53, 'NCL': 53, 'PYF': 53,
}
SUBREGION = {11: 'Northern Europe', 12: 'Southern Europe', 13: 'Western Europe', 14: 'Eastern Europe',
             15: 'Northern Africa', 16: 'Western Africa', 17: 'Eastern Africa', 18: 'Middle Africa',
             19: 'Southern Africa', 21: 'Northern America', 22: 'Central America', 23: 'Caribbean',
             24: 'South America', 30: 'Eastern Asia', 34: 'Southern Asia', 35: 'South-eastern Asia',
             53: 'Oceania', 143: 'Central Asia', 145: 'Western Asia'}

df = pd.read_csv('../outputs/data_prep/panel_final.csv')
df = df[(df['orig'] != df['dest']) & (df['year'] <= T)]
if COUNTRIES is not None:
    df = df[df['orig'].isin(COUNTRIES) & df['dest'].isin(COUNTRIES)]
df = df.sort_values(['orig', 'dest', 'year']).reset_index(drop=True)
df['dyad'] = df['orig'] + '_' + df['dest']
df['rid'] = np.arange(len(df))
m49 = df['orig'].map(ISO3_TO_M49)
M49_TO_K = {m: k + 1 for k, m in enumerate(sorted(m49.unique()))}
df['cl'] = m49.map(M49_TO_K)                                   # M49 sub-region of the origin
df['log_P_it'], df['log_P_jt'] = np.log(df['P_it']), np.log(df['P_jt'])
df['log_flow_lag_clean'] = df['log_flow_lag'].fillna(0.0)

g = df.groupby('dyad')
df['flow_momentum'] = (g['log_flow'].shift(1) - g['log_flow'].shift(2)).fillna(0.0)
df['flow_prev'] = g['flow'].shift(1).where(g['year'].shift(1) == df['year'] - 5)   # persistence benchmark
for k in (1, 2):                                               # row of the same dyad k windows earlier
    df[f'rid_l{k}'] = g['rid'].shift(k).where(g['year'].shift(k) == df['year'] - 5 * k)
df['has_l2'] = (g['log_flow'].shift(2).notna() & df['rid_l2'].notna()).astype(int)

# Network features from the corridors open in the previous window
act = df['is_mig_lag'].eq(1)
df['out_degree_o'] = act.groupby([df['orig'], df['year']]).transform('sum').astype(float)
df['in_degree_d'] = act.groupby([df['dest'], df['year']]).transform('sum').astype(float)
df['transitivity_proxy'] = df['out_degree_o'].fillna(0) * df['in_degree_d'].fillna(0)
pays = sorted(set(df['orig']) | set(df['dest']))
cid = {p: i + 1 for i, p in enumerate(pays)}
oi, dj = df['orig'].map(cid).values - 1, df['dest'].map(cid).values - 1
a2 = np.zeros(len(df))
for y, idx in df.groupby('year').indices.items():               # paths i -> k -> j open in the previous window
    A = np.zeros((len(pays), len(pays)))
    a = idx[act.values[idx]]
    A[oi[a], dj[a]] = 1.0
    a2[idx] = (A @ A)[oi[idx], dj[idx]]
df['A2_log'] = np.log1p(a2)

# Covariates of BART (existence) and of the volume mean
BART_VARS = ['log_D_ij', 'log_D_ij_sq', 'OL_ij', 'COL_ij', 'LB_ij',
             'log_gdpcap_k_o_lag5', 'log_gdpcap_k_d_lag5', 'log_P_it', 'log_P_jt',
             'PSR_i', 'PSR_j', 'IMR_it', 'IMR_jt', 'urban_wup_it', 'urban_wup_jt',
             'LL_i', 'LL_j', 'LA_i', 'LA_j', 'v2x_polyarchy_o_lag5', 'v2x_polyarchy_d_lag5',
             'is_mig_lag', 'log_stock_lag', 'transitivity_proxy', 'A2_log', 'flow_momentum',
             'log_stock_dia', 'log_stock_rev']
X_VOL = ['log_D_ij', 'LB_ij', 'OL_ij', 'COL_ij', 'v2x_polyarchy_o_lag5', 'intensity_level_o_lag5',
         'log_gdpcap_k_o_lag5', 'log_stock_dia', 'log_stock_rev']
# X_VOL += ['ml_grav']                   # XGBoost gravity term (cell below), left out
BINARY = {'LB_ij', 'OL_ij', 'COL_ij'}                                    # not standardised
BETA_PRIOR = {'log_D_ij': (-0.5, 0.5), 'ml_grav': (0.0, 2.5)}            # (mean, sd); others N(0, 1)

# Volume training: positive flows of the windows 1995 to T - 5; forecast rows: calibration windows and T
vol_m = df['year'].between(1995, T - 5) & df['flow'].gt(0) & df['is_mig_lag'].isin([0, 1])
vol_m &= df[[c for c in X_VOL if c != 'ml_grav']].notna().all(axis=1)
test_m = df['year'].isin(CAL_YEARS + [T])

print(f"{df['orig'].nunique()} countries, {df['dyad'].nunique():,} dyads, {len(M49_TO_K)} M49 sub-regions")
print(f"Volume training: {vol_m.sum():,} positive flows, of which {(vol_m & df['is_mig_lag'].eq(0)).sum():,} "
      "in corridors closed in the previous window")
print(f"Forecast rows: {test_m.sum():,} (calibration {CAL_YEARS}, forecast {T})")
print(f"BART covariates ({len(BART_VARS)}): {', '.join(BART_VARS)}")
print(f"Volume covariates ({len(X_VOL)}): {', '.join(X_VOL)}")

### XGBoost gravity term

$$
\hat m_{ijt}=\hat f_{-g(ij)}(x_{ijt})
$$

$\hat f_{-g}$: XGBoost regression of $\log Y$ on the covariates without history (no lagged flow, stock or network
feature), fitted on the training flows of the dyads outside fold $g$ (five folds by dyad). Dyads absent from training
get the mean of the five models. Left out: this cell runs only if `'ml_grav'` is added to `X_VOL` above.

In [ ]:
if 'ml_grav' in X_VOL:
    import xgboost as xgb
    from sklearn.model_selection import GroupKFold
    HISTORY = {'is_mig_lag', 'log_stock_lag', 'transitivity_proxy', 'A2_log', 'flow_momentum',
               'log_stock_dia', 'log_stock_rev'}                         # left out: persistence is the AR part's job
    ML_VARS = [v for v in BART_VARS if v not in HISTORY]
    tr = df[vol_m]
    fold, models = pd.Series(-1, index=df['dyad'].unique()), []
    for f, (i, j) in enumerate(GroupKFold(5).split(tr, groups=tr['dyad'])):
        m = xgb.XGBRegressor(n_estimators=400, max_depth=6, learning_rate=0.05, subsample=0.8,
                             colsample_bytree=0.8, random_state=SEED, n_jobs=4)
        models.append(m.fit(tr[ML_VARS].iloc[i], np.log(tr['flow'].iloc[i])))
        fold[tr['dyad'].iloc[j].unique()] = f
    pred = np.column_stack([m.predict(df[ML_VARS]) for m in models])
    f = df['dyad'].map(fold).values
    df['ml_grav'] = np.where(f >= 0, pred[np.arange(len(df)), np.maximum(f, 0)], pred.mean(1))
    r2 = np.corrcoef(df.loc[vol_m, 'ml_grav'], np.log(df.loc[vol_m, 'flow']))[0, 1] ** 2
    print(f"XGBoost gravity term: out-of-fold R2 on log volume = {r2:.3f}")

## Volume: zero-truncated negative binomial (Stan)

$$
P(V_{ijt}=v)=\frac{\mathrm{NB}(v\mid m_{ijt},\phi_{ijt})}{1-\left(\frac{\phi_{ijt}}{\phi_{ijt}+m_{ijt}}\right)^{\phi_{ijt}}},\quad v\ge 1,
\qquad
\mathrm{NB}(v\mid m,\phi)=\frac{\Gamma(v+\phi)}{\Gamma(\phi)\,v!}\left(\frac{\phi}{\phi+m}\right)^{\phi}\left(\frac{m}{\phi+m}\right)^{v}
$$

$$
m_{ijt}=e^{\eta_{ijt}},\qquad
\mathbb{E}[V_{ijt}]=\frac{e^{\eta_{ijt}}}{1-\left(\frac{\phi_{ijt}}{\phi_{ijt}+e^{\eta_{ijt}}}\right)^{\phi_{ijt}}},\qquad
\operatorname{Var}_{\mathrm{NB}}=m+\frac{m^{2}}{\phi}
$$

**Mean**

$$
\mu_{ijt}=\alpha_{it}+\gamma_{jt}+x_{ijt}^{\top}\beta
$$

$$
\eta_{ijt}=
\begin{cases}
\mu_{ijt}+\rho_{ij}\left(\bar L_{ijt}-\bar\mu_{ijt}\right)+\xi_{it}+\zeta_{jt}, & Y_{ij,t-5}>0\\
\mu_{ijt}+\kappa_{k}+\xi_{it}+\zeta_{jt}, & Y_{ij,t-5}=0
\end{cases}
$$

$$
\bar L_{ijt}=\omega\log Y_{ij,t-5}+(1-\omega)\log Y_{ij,t-10},\qquad
\bar\mu_{ijt}=\omega\,\mu_{ij,t-5}+(1-\omega)\,\mu_{ij,t-10}
$$

Both with $\omega=1$ when $Y_{ij,t-10}$ is $0$ or missing; $\bar\mu_{ijt}=\mu_{ijt}$ when the covariates of $t-5$ are
missing (1990, a few countries).

**Country effects and country-period shocks** (code: `em` origin, `at` destination)

$$
\alpha_{it}=c+\tau_{\alpha}a_{i}+\theta_{\alpha}z_{it},\qquad
\gamma_{jt}=\tau_{\gamma}g_{j}+\theta_{\gamma}z_{jt},\qquad
\xi_{it}\sim\mathcal N(0,\sigma_{\xi}^{2}),\qquad \zeta_{jt}\sim\mathcal N(0,\sigma_{\zeta}^{2})
$$

$z_{it}$: log GDP of year $t-1$ in constant prices, standardised across countries for each $t$. The shocks are
independent, one per country and training window; window $T$: $\xi_{iT}$ and $\zeta_{jT}$ are drawn for each
posterior draw, so that all the corridors of a country share them.

**Persistence, entry, dispersion** ($k$: M49 sub-region of the origin)

$$
\rho_{ij}=\tanh\left(r_k+\tau_\rho\,\epsilon_{ij}\right),\qquad r_k=\mu_r+\sigma_r\tilde r_k,\qquad
\kappa_k=\mu_\kappa+\sigma_\kappa\tilde\kappa_k,\qquad \omega=\operatorname{logit}^{-1}(\tilde\omega)
$$

$$
\phi_{ijt}=\phi_k\exp\left(\delta_\phi s_{ijt}+\tau_\phi\,\epsilon'_{ij}\right),\qquad
\log\phi_k\sim\mathcal N\left(\log\phi_{\mathrm{global}},\,0.5^{2}\right)
$$

$s_{ijt}$: $\log Y_{ij,t-5}$ ($0$ if $Y_{ij,t-5}=0$), standardised over the training flows. Dyads absent from
training: $\epsilon_{ij},\epsilon'_{ij}\sim\mathcal N(0,1)$ drawn for each posterior draw.

**Priors**, weakly informative; for instance

$$
\beta_{\log D}\sim\mathcal N(-0.5,0.5^2),\ \text{other }\beta\sim\mathcal N(0,1),\qquad
\tau_\alpha,\tau_\gamma\sim\mathcal N^{+}(0,0.25^2),\qquad \sigma_\xi,\sigma_\zeta\sim\mathcal N^{+}(0,0.5^2)
$$

($\mathcal N^{+}$: half-normal); $a_i$, $g_j$ and the other standardised effects are $\mathcal N(0,1)$. All priors are in
`stan/hurdle_volume.stan`.

Covariates standardised over the training flows, except the binary ones (border, language, colonial link).
Likelihood: the positive flows of the windows 1995 to $T-5$.

In [ ]:
vol, test = df[vol_m].copy(), df[test_m].copy()
for fr in (vol, test):
    fr['is_ar'] = fr['is_mig_lag'].eq(1).astype(int)
dyads_v = sorted(vol['dyad'].unique())
dyad_v = {d: i + 1 for i, d in enumerate(dyads_v)}
years_p = list(range(1990, T + 1, 5))   # P periods 1990, ..., T - 5, then T
P = len(years_p) - 1
x_ok = df[X_VOL].notna().all(axis=1).values


def cells(fr):
    """Rows (cells) whose structural mean enters each row: windows t, t - 5 and t - 10 (the previous one if
    unavailable), and whether the t - 10 cell is used."""
    c0 = fr['rid'].values
    r1 = fr['rid_l1'].fillna(-1).astype(int).values
    r2 = fr['rid_l2'].fillna(-1).astype(int).values
    ok1 = (r1 >= 0) & x_ok[np.maximum(r1, 0)] & fr['is_ar'].eq(1).values
    h2 = ok1 & fr['has_l2'].eq(1).values & (r2 >= 0) & x_ok[np.maximum(r2, 0)]
    c1 = np.where(ok1, r1, c0)
    return c0, c1, np.where(h2, r2, c1), h2.astype(int)


tr_c, te_c = cells(vol), cells(test)
cell_rows = pd.unique(np.concatenate(tr_c[:3]))
n_cell_tr = len(cell_rows)                 # training cells come first
cell_rows = np.concatenate([cell_rows, np.setdiff1d(np.concatenate(te_c[:3]), cell_rows)])
cell_ix = pd.Series(np.arange(1, len(cell_rows) + 1), index=cell_rows)
cdf = df.iloc[cell_rows]
mu_x, sd_x = vol[X_VOL].mean(), vol[X_VOL].std(ddof=0).clip(lower=1e-8)
for c in BINARY & set(X_VOL):
    mu_x[c], sd_x[c] = 0.0, 1.0
X = ((cdf[X_VOL] - mu_x) / sd_x).values
per = cdf['year'].map({y: p for p, y in enumerate(years_p)}).values
li_o = per * len(pays) + cdf['orig'].map(cid).values
li_d = per * len(pays) + cdf['dest'].map(cid).values

# z_it: log GDP of year t - 1 in constant prices, standardised across countries for each window
lg = df.groupby(['orig', 'year'])['log_gdp_k_o_lag1'].first().unstack().reindex(index=pays, columns=years_p)
lg = lg.T.interpolate(limit_direction='both').T                         # gaps: nearest years of the country
Z = ((lg - lg.mean()) / lg.std(ddof=0)).values

# s: log Y_{t-5} (0 if the corridor was closed), standardised over the training flows
ly = 'log_flow_lag_clean'
s_tr = ((vol[ly] - vol[ly].mean()) / vol[ly].std(ddof=0)).values
s_te = ((test[ly] - vol[ly].mean()) / vol[ly].std(ddof=0)).values


# country-period shocks: one per country and training window with observations, then T
q_years = sorted(vol['year'].unique())
q_of = {y: q for q, y in enumerate(q_years)} | {T: len(q_years)}
x_ix = lambda fr, side: (fr['year'].map(q_of) * len(pays) + fr[side].map(cid)).values


def mom(fr, h2):
    """log Y_{t-5} - log Y_{t-10}, 0 when the t - 10 cell is not used."""
    return np.where(h2 == 1, fr['flow_momentum'].values, 0.0)


prior = np.array([BETA_PRIOR.get(v, (0.0, 1.0)) for v in X_VOL])
stan_data = dict(
    N_c=len(pays), P=P, M=len(M49_TO_K), K=len(X_VOL), Z=Z, beta_mu=prior[:, 0], beta_sd=prior[:, 1],
    N_cell=len(cell_rows), N_cell_tr=n_cell_tr, li_o=li_o, li_d=li_d, X=X,
    N=len(vol), D=len(dyads_v), y=vol['flow'].astype(int).values, dyad=vol['dyad'].map(dyad_v).values,
    cl=vol.groupby('dyad')['cl'].first().reindex(dyads_v).values,
    c0=cell_ix.loc[tr_c[0]].values, c1=cell_ix.loc[tr_c[1]].values, c2=cell_ix.loc[tr_c[2]].values,
    is_ar=vol['is_ar'].values, has_l2=tr_c[3], ly1=vol[ly].values, mom=mom(vol, tr_c[3]), s=s_tr,
    Q=len(q_years), xo=x_ix(vol, 'orig'), xd=x_ix(vol, 'dest'),
    N_t=len(test), dyad_t=test['dyad'].map(dyad_v).fillna(0).astype(int).values, cl_t=test['cl'].values,
    c0_t=cell_ix.loc[te_c[0]].values, c1_t=cell_ix.loc[te_c[1]].values, c2_t=cell_ix.loc[te_c[2]].values,
    is_ar_t=test['is_ar'].values, has_l2_t=te_c[3], ly1_t=test[ly].values, mom_t=mom(test, te_c[3]), s_t=s_te,
    xo_t=x_ix(test, 'orig'), xd_t=x_ix(test, 'dest'),
)
assert list(cdf[X_VOL].columns) == X_VOL
assert all(np.isfinite(np.asarray(v, float)).all() for v in stan_data.values()), "NaN or Inf in stan_data"
unseen = test['year'].eq(T).values & (stan_data['dyad_t'] == 0)
print(f"Stan data: {len(vol):,} observations, {len(dyads_v):,} dyads, {len(cell_rows):,} cells | dyads of {T} unseen "
      f"in training: {unseen.sum():,}, of which open {(unseen & test['flow'].gt(0).values).sum():,}")

## Existence: probit BART with warm start (stochtree)

$$
p_{ijt}=P(Y_{ijt}>0)=\Phi\Big(c_0+\sum_{h=1}^{200}g\big(x_{ijt};\mathcal T_h,\mathcal M_h\big)\Big),\qquad
c_0=\Phi^{-1}\big(\bar y_{\mathrm{train}}\big)
$$

$\bar y_{\mathrm{train}}$: share of open corridors in the training rows. Tree prior: a node at depth $d$ splits with
probability $0.95\,(1+d)^{-2}$ (depth at most 10, at least 5 rows per leaf); leaf values
$\mathcal N\big(0,\,(3/(k\sqrt{200}))^{2}\big)$ with $k=2$.

Chains $c=1,\dots,16$: 10 grow-from-root (XBART) sweeps from single-leaf trees, then 100 burn-in and 300 kept iterations
of the BART sampler; seed $42+c$. Window $y\in\{T-10,\,T-5,\,T\}$ is forecast by a BART trained on the windows 1995 to
$y-5$. To reuse the draws of a finished run instead of fitting: `zb, N_h, bart_imp, t_bart = saved_bart_draws(OUT)`.

In [ ]:
def bart_chain(Xb, yb, Xt, seed, prior, ngfr, nskip, ndpost):
    """One warm-start BART chain: ngfr XBART sweeps from scratch, then nskip burn-in and ndpost draws of the BART
    sampler. Probit-scale draws (draws, rows) and the number of splits on each covariate."""
    import warnings
    from stochtree import BARTModel, OutcomeModel
    warnings.filterwarnings('ignore', message='Global error variance will not be sampled')   # fixed at 1 with probit
    m = BARTModel()
    m.sample(X_train=Xb, y_train=yb, X_test=Xt, num_gfr=ngfr, num_burnin=nskip, num_mcmc=ndpost,
             general_params={'outcome_model': OutcomeModel(outcome='binary', link='probit'), 'random_seed': seed,
                             'num_threads': 1},
             mean_forest_params=prior)
    return m.y_hat_test.T.astype(np.float32), m.forest_container_mean.get_overall_split_counts(Xb.shape[1])


def bart_draws(last_year, rows):
    """BART trained on the windows 1995 to last_year, one process per chain: probit-scale draws for `rows`
    (chains, draws, rows), the number of training rows, and the share of the tree splits on each covariate."""
    tr = df[df['year'].between(1995, last_year)].dropna(subset=BART_VARS)
    assert rows[BART_VARS].notna().all().all()
    t0 = time.time()
    Xb, yb = tr[BART_VARS].values.astype(float), tr['flow'].gt(0).values.astype(float)
    Xt = rows[BART_VARS].values.astype(float)
    prior = {'num_trees': NTREE, 'alpha': BASE, 'beta': POWER, 'sample_sigma2_leaf': False,
             'sigma2_leaf_init': (3 / (K_BART * np.sqrt(NTREE))) ** 2}
    out = Parallel(n_jobs=min(BART_CHAINS, cpu_count()), max_nbytes=None)(
        delayed(bart_chain)(Xb, yb, Xt, SEED_BART + c, prior, BART_GFR, BART_BURN, BART_DRAWS) for c in range(BART_CHAINS))
    splits = np.sum([n for _, n in out], axis=0).astype(float)
    print(f"BART trained on 1995-{last_year}: {len(tr):,} rows -> {len(rows):,} forecasts ({time.time() - t0:.0f} s)")
    return np.stack([z for z, _ in out]), len(tr), pd.Series(splits / splits.sum(), index=BART_VARS)


t0 = time.time()
yrs = test['year'].values
zb = np.empty((BART_CHAINS, BART_DRAWS, len(test)), np.float32)
for y in CAL_YEARS + [T]:                  # each window forecast by a BART trained before it
    zb[:, :, yrs == y], N_h, bart_imp = bart_draws(y - 5, test[yrs == y])
t_bart = time.time() - t0
(OUT / 'bart').mkdir(exist_ok=True)
np.save(OUT / 'bart' / 'zb.npy', zb)
(OUT / 'bart' / 'info.json').write_text(json.dumps(
    {'N_h': N_h, 'importance': {k: float(v) for k, v in bart_imp.items()}, 't_bart': t_bart}, indent=1))

In [ ]:
pb = ndtr(zb.reshape(-1, len(test)))           # draws of p, chain after chain
p_mean, p_med = pb.mean(0, dtype=float), np.median(pb, axis=0)
cal, ev = test['year'].isin(CAL_YEARS).values, test['year'].eq(T).values
yv = test['flow'].values


def existence_scores(m):
    p, y = np.clip(p_mean[m], 1e-9, 1 - 1e-9), yv[m] > 0
    return {'AUC': roc_auc_score(y, p), 'Brier': np.mean((p - y) ** 2),
            'log score': np.where(y, np.log(p), np.log1p(-p)).mean()}


print("Existence scores (p = posterior mean of the BART probability):")
display(pd.DataFrame({f'calibration {CAL_YEARS}': existence_scores(cal), f'forecast {T}': existence_scores(ev)}).T.round(4))
imp = pd.DataFrame({'share of splits (%)': 100 * bart_imp, 'relative to uniform': bart_imp * len(bart_imp)})
print(f"BART covariates by share of the splits, BART of {T} (uniform share {100 / len(bart_imp):.1f}%):")
display(imp.sort_values('share of splits (%)', ascending=False).round(2))

sub_b = np.random.default_rng(0).choice(np.flatnonzero(ev), min(2000, ev.sum()), replace=False)
rb, eb, etb = diagnostics(zb[:, :, sub_b].astype(float))
bart_checks = [('BART: share of corridors with R-hat > 1.01', (rb > 1.01).mean(), '< 0.05', (rb > 1.01).mean() < 0.05),
               ('BART: median R-hat', np.median(rb), '< 1.01', np.median(rb) < 1.01),
               ('BART: median bulk ESS', np.median(eb), '> 400', np.median(eb) > 400),
               ('BART: median tail ESS', np.median(etb), '> 400', np.median(etb) > 400)]
bart_chain = bart_agreement(zb[:, :, ev], yv[ev] > 0)
print(f"BART chains, {len(sub_b):,} random corridors of {T}: median R-hat {np.median(rb):.2f} (for information) | "
      f"log score of p {bart_chain['log score']:.4f}, of single chains {bart_chain['log score, worst chain']:.4f} "
      f"to {bart_chain['log score, best chain']:.4f} | Monte Carlo error of p: median {bart_chain['se of p, median']:.4f}, "
      f"99th percentile {bart_chain['se of p, 99th percentile']:.4f}")

## Volume: sampling

To reuse the draws of a finished run instead of sampling, replace the next cell by
`CSV_FILES, t_stan = saved_stan_draws(OUT)`.

In [ ]:
t0 = time.time()
STAN_FILE = '../stan/hurdle_volume.stan'
(OUT / 'stan').mkdir(exist_ok=True)
np.savez_compressed(OUT / 'stan' / 'stan_data.npz', **stan_data)
shutil.copy(STAN_FILE, OUT / 'stan')                                     # the model of these draws
print(f"Stan sampling: {N_CHAINS} chains x ({ITER_WARMUP} warm-up + {ITER_SAMPLING} draws) on {len(vol):,} observations...")
fit = CmdStanModel(stan_file=STAN_FILE).sample(
    data=stan_data, chains=N_CHAINS, parallel_chains=N_CHAINS, iter_warmup=ITER_WARMUP, iter_sampling=ITER_SAMPLING,
    seed=SEED, inits=0.1, adapt_delta=ADAPT_DELTA, max_treedepth=MAX_TREEDEPTH, sig_figs=4,
    output_dir=OUT / 'stan', show_progress=False)
CSV_FILES, t_stan = fit.runset.csv_files, time.time() - t0

In [ ]:
zb = None                                  # free the probit-scale BART draws: pb holds what follows uses
t0 = time.time()
gq, cols = read_stan_csv(CSV_FILES, {'treedepth__', 'divergent__', 'eta_t', 'phi_t'}, ITER_SAMPLING)   # one pass
assert cols == (['treedepth__', 'divergent__'] + [f'eta_t.{i}' for i in range(1, len(test) + 1)]
                + [f'phi_t.{i}' for i in range(1, len(test) + 1)])
sampler = pd.DataFrame(gq[:, :2], columns=cols[:2])
eta, phi = gq[:, 2:2 + len(test)], gq[:, 2 + len(test):]                 # views of gq, no copy
bad = ~(np.isfinite(eta).all(1) & np.isfinite(phi).all(1))
assert not bad.any(), f"{bad.sum()} Stan draws with a non-finite forecast: inspect the run before using it"
n_div = int(sampler['divergent__'].sum())
print(f"Stan: {t_stan / 60:.1f} min, draws read in {(time.time() - t0) / 60:.1f} min | {len(eta):,} draws | "
      f"divergent transitions: {n_div} | iterations at the maximum tree depth: "
      f"{(sampler['treedepth__'] >= MAX_TREEDEPTH).mean() * 100:.1f}%")

## Convergence

Each chain is split in two halves: $J$ sequences of $N$ draws. Draws are replaced by the normal scores of their ranks $r$:
$z=\Phi^{-1}\big((r-3/8)/(JN+1/4)\big)$.

$$
\hat R=\sqrt{\frac{\frac{N-1}{N}W+B}{W}},\qquad W=\frac1J\sum_{j=1}^{J}s_j^2,\qquad B=\frac{1}{J-1}\sum_{j=1}^{J}\big(\bar z_j-\bar z\big)^2
$$

$$
\mathrm{ESS}=\frac{JN}{\hat\tau},\qquad \hat\tau=-1+2\sum_{l=0}^{L}\big(\hat\rho_{2l}+\hat\rho_{2l+1}\big),\qquad
\hat\rho_t=1-\frac{W-\frac1J\sum_j\hat\gamma_{j,t}}{\frac{N-1}{N}W+B}
$$

$s_j^2$, $\bar z_j$, $\hat\gamma_{j,t}$: variance, mean and lag-$t$ autocovariance of sequence $j$. $L$: last pair before a
negative sum; the pair sums are made non-increasing (Geyer). Bulk ESS: on $z$. Tail ESS: the smaller ESS of
$\mathbb 1\{\theta\le q_{0.05}\}$ and $\mathbb 1\{\theta\le q_{0.95}\}$. Targets: $\hat R<1.01$, ESS $>400$.

Monte Carlo error of $\bar p$, from the means $\bar p^{(c)}$ of the $C=16$ BART chains:

$$
\mathrm{se}\big(\bar p_{ijt}\big)=\frac{\mathrm{sd}_c\big(\bar p^{(c)}_{ijt}\big)}{\sqrt{C}}
$$

In [ ]:
PARS = ['lp__', 'c', 'theta_em', 'theta_at', 'tau_em', 'tau_at', 'sig_xi_em', 'sig_xi_at', 'beta',
        'mu_r', 'sigma_r', 'tau_rho', 'mu_kappa', 'sigma_kappa', 'omega', 'phi_global', 'tau_phi', 'delta_phi',
        'r', 'kappa', 'phi_k']
GROUPS = {'a_em': ('origin effects', len(pays)), 'a_at': ('destination effects', len(pays)),
          'xi_em': ('origin-period shocks', len(pays) * stan_data['Q']),
          'xi_at': ('destination-period shocks', len(pays) * stan_data['Q']),
          'eps': ('dyadic persistence', stan_data['D']), 'eps2': ('dyadic dispersion', stan_data['D'])}
sub_d = np.random.default_rng(2).choice(stan_data['D'], min(2000, stan_data['D']), replace=False) + 1
names = set(PARS) | set(GROUPS) - {'eps', 'eps2'} | {f'eps{v}.{i}' for v in ('', '2') for i in sub_d}
a, cols = read_stan_csv(CSV_FILES, names, ITER_SAMPLING)                      # one pass over the CSV files
arr, cols = a.reshape(N_CHAINS, ITER_SAMPLING, -1), pd.Index(cols)
var = cols.str.split('.').str[0]
rh, es, et = diagnostics(arr)
q = np.percentile(a, [2.5, 50, 97.5], axis=0)
tab = pd.DataFrame({'median': q[1], 'q2.5': q[0], 'q97.5': q[2], 'P(> 0)': (a > 0).mean(0),
                    '95% excludes 0': (q[0] > 0) | (q[2] < 0), 'R-hat': rh, 'bulk ESS': es, 'tail ESS': et}, index=cols)
del a, arr
k49 = {k: m for m, k in M49_TO_K.items()}
in_pars, regional = var.isin(PARS), var.isin(['r', 'kappa', 'phi_k'])
label = lambda c: f"beta[{X_VOL[int(c.split('.')[1]) - 1]}]" if c.startswith('beta.') else c
params_tab = tab[in_pars & ~regional].rename(index=lambda c: label(c) if c.startswith('beta.') else c.split('.')[0])
print("Global parameters (beta: standardised covariates):")
display(params_tab.round(3))
reg = pd.DataFrame({'region': [SUBREGION.get(k49[k], k49[k]) for k in sorted(k49)],
                    'rho_k = tanh(r_k)': np.tanh(tab.loc[[f'r.{k}' for k in sorted(k49)], 'median'].values),
                    'kappa_k': tab.loc[[f'kappa.{k}' for k in sorted(k49)], 'median'].values,
                    'phi_k': tab.loc[[f'phi_k.{k}' for k in sorted(k49)], 'median'].values,
                    'worst R-hat': tab[regional]['R-hat'].values.reshape(3, -1).max(0),
                    'lowest ESS': np.minimum(tab[regional]['bulk ESS'], tab[regional]['tail ESS']).values.reshape(3, -1).min(0)})
print("Regional parameters (posterior medians), M49 sub-region of origin:")
display(reg.set_index('region').round(3))
groups = {}
for g, (name, n_all) in GROUPS.items():
    m = np.asarray(var == g)
    groups[f'{g}: {name}'] = {'checked': f"{m.sum():,} of {n_all:,}",
                              'share R-hat > 1.01': round((rh[m] > 1.01).mean(), 3), 'max R-hat': round(rh[m].max(), 3),
                              'median bulk ESS': round(np.median(es[m])), 'min bulk ESS': round(es[m].min()),
                              'min tail ESS': round(et[m].min()), 'worst': cols[m][np.argmax(rh[m])]}
print("High-dimensional parameters (dyadic ones on 2,000 random dyads):")
display(pd.DataFrame(groups).T)
print("The 10 parameters with the highest R-hat:")
display(tab.sort_values('R-hat', ascending=False).head(10).rename(index=label).round(3))

# Forecasts: eta and log phi of 2,000 random corridors of window T
sub_s = np.random.default_rng(1).choice(np.flatnonzero(ev), min(2000, ev.sum()), replace=False)
fc = np.concatenate([eta[:, sub_s], np.log(phi[:, sub_s])], axis=1).reshape(N_CHAINS, -1, 2 * len(sub_s))
rf, ef, etf = diagnostics(fc)
p_rh, p_es, p_et = rh[in_pars], es[in_pars], et[in_pars]
hd = np.asarray(var.isin(list(GROUPS)))
stan_checks = [('Stan: divergent transitions', n_div, '= 0', n_div == 0),
               ('Stan: max R-hat, parameters above', p_rh.max(), '< 1.01', p_rh.max() < 1.01),
               ('Stan: min bulk ESS, parameters above', p_es.min(), '> 400', p_es.min() > 400),
               ('Stan: min tail ESS, parameters above', p_et.min(), '> 400', p_et.min() > 400),
               ('Stan: share of country and dyadic parameters with R-hat > 1.01', (rh[hd] > 1.01).mean(), '< 0.05',
                (rh[hd] > 1.01).mean() < 0.05),
               ('Stan: share of forecasts with R-hat > 1.01', (rf > 1.01).mean(), '< 0.05', (rf > 1.01).mean() < 0.05),
               ('Stan: median bulk ESS, forecasts', np.median(ef), '> 400', np.median(ef) > 400),
               ('Stan: median tail ESS, forecasts', np.median(etf), '> 400', np.median(etf) > 400)]
print(f"Convergence checks (forecasts: eta and log phi of {len(sub_s):,} random corridors of {T}; BART: for information):")
display(pd.DataFrame(stan_checks + bart_checks, columns=['check', 'value', 'target', 'ok']).round(3))

## Forecasts and scores

Draw $s=1,\dots,S$ ($S=4{,}800$) pairs Stan draw $s$ with one BART draw:

$$
Y^{(s)}_{ijt}=B^{(s)}_{ijt}\,V^{(s)}_{ijt},\qquad B^{(s)}_{ijt}\sim\mathrm{Bernoulli}\big(p^{(s)}_{ijt}\big),\qquad
V^{(s)}_{ijt}\sim\mathrm{ZTNB}\big(e^{\eta^{(s)}_{ijt}},\phi^{(s)}_{ijt}\big),\qquad \bar p_{ijt}=\frac1S\sum_s p^{(s)}_{ijt}
$$

Loss and point forecasts ($\lambda=1$):

$$
L_\lambda(y,\hat y)=|y-\hat y|\;w(y),\qquad w(y)=1+\frac{100\lambda}{y+1}
$$

- median rule: $\hat y=\operatorname{median}_s V^{(s)}$ if $\bar p\ge h_k$, else $0$; $h_k$ minimises $\sum L_\lambda(y,\hat y)$
  over the corridors of sub-region $k$ in the calibration windows;
- weighted median rule: the same with the median of $V$ weighted by $w(V)$;
- Bayes action, no threshold: $\hat y=\arg\min_a\mathbb E\big[L_\lambda(Y,a)\big]$, the median of the predictive
  distribution weighted by $w$ (mass $1-\bar p$ at $0$).

Scores on the $n$ corridors of window $T$ ($n_+$ open):

$$
\mathrm{Brier}=\frac1n\sum\big(\bar p-\mathbb 1\{y>0\}\big)^2,\qquad
\mathrm{LS}_{\mathrm{exist}}=\frac1n\sum\Big[\mathbb 1\{y>0\}\log\bar p+\mathbb 1\{y=0\}\log(1-\bar p)\Big]
$$

$$
\mathrm{LS}_{\mathrm{vol}}=\frac1{n_+}\sum_{y>0}\log\hat f(y),\qquad
\hat f(y)=\frac1S\sum_s\mathrm{ZTNB}\big(y\mid e^{\eta^{(s)}},\phi^{(s)}\big)
$$

$$
\mathrm{LS}_{\mathrm{total}}=\frac1n\sum\log\hat P(Y=y),\qquad \hat P(Y=0)=1-\bar p,\qquad \hat P(Y=y)=\bar p\,\hat f(y)\ \ (y>0)
$$

$$
\mathrm{CRPS}=\frac1n\sum\Big(\mathbb E|Y-y|-\tfrac12\,\mathbb E|Y-Y'|\Big),\qquad Y,Y'\ \text{independent predictive draws}
$$

$\mathrm{MAE}=\frac1n\sum|y-\hat y|$, $\mathrm{MAPE}=\frac{100}{n}\sum\frac{|y-\hat y|}{y+1}$; FP: $\hat y>0=y$; FN:
$\hat y=0<y$. Coverage: share of the corridors, and of the countries for the totals $\sum_j Y_{ij}$ (Out),
$\sum_i Y_{ij}$ (In) and In − Out (Net), inside the 2.5–97.5% predictive interval.

In [ ]:
t0 = time.time()
S = len(eta)
idx_b = np.random.RandomState(0).choice(len(pb), S, replace=S > len(pb))   # one BART draw per Stan draw
rng = np.random.default_rng(SEED)
R = {k: np.full(len(test), np.nan) for k in ('v_med', 'v_wmed', 'q_lo', 'q_hi', 'bayes', 'crps', 'lpmf')}
co, cd = test['orig'].map(cid).values - 1, test['dest'].map(cid).values - 1
OUT_draws, IN_draws = np.zeros((S, len(pays))), np.zeros((S, len(pays)))   # emigrants, immigrants of window T
eye = np.eye(len(pays))
for a in range(0, len(test), 2000):
    j = slice(a, a + 2000)
    V = ztnb_draw(eta[:, j], phi[:, j], rng)
    w = 1 + 100 * LAMBDA / (V + 1.0)
    pbj = pb[idx_b, j]
    Yt = np.where(rng.random(V.shape) < pbj, V, 0)                      # Bernoulli(p) x ZTNB
    R['v_med'][j], R['v_wmed'][j] = np.median(V, 0), wmedian(V, w)
    R['q_lo'][j], R['q_hi'][j] = np.percentile(Yt, [2.5, 97.5], axis=0)
    R['bayes'][j], R['crps'][j] = bayes_action(V, pbj, w), crps(Yt, yv[j])
    pos = np.flatnonzero(yv[j] > 0)
    R['lpmf'][a + pos] = ztnb_logpmf(yv[j][pos], eta[:, j][:, pos], phi[:, j][:, pos])
    e = ev[j]
    OUT_draws += Yt[:, e] @ eye[co[j][e]]
    IN_draws += Yt[:, e] @ eye[cd[j][e]]
assert np.allclose(OUT_draws.sum(1), IN_draws.sum(1)), "country totals do not add up"

# Point forecasts of window T: thresholds per sub-region from the calibration windows (median and weighted median
# rules), Bayes action without threshold, and persistence Y_T = Y_{T-5}
cl_te = test['cl'].values
y, yb = yv[ev], yv[ev] > 0
pred = {}
for rule, v_pt in [('median', R['v_med']), ('wmedian', R['v_wmed'])]:
    thr = thresholds(p_mean[cal], yv[cal], v_pt[cal], cl_te[cal], LAMBDA)
    t_def = float(np.median(list(thr.values())))
    pred[rule] = np.where(p_mean[ev] >= np.array([thr.get(k, t_def) for k in cl_te[ev]]), v_pt[ev], 0.0)
pred['bayes'] = R['bayes'][ev]
pred['persistence'] = test['flow_prev'].fillna(0).values[ev]
point = pd.DataFrame.from_dict({rule: {'FP': int(((p > 0) & ~yb).sum()), 'FN': int(((p == 0) & yb).sum()),
                                        'MAE': np.abs(y - p).mean(), 'MAPE (%)': (np.abs(y - p) / (y + 1)).mean() * 100}
                                 for rule, p in pred.items()}, orient='index')

inside = (y >= R['q_lo'][ev]) & (y <= R['q_hi'][ev])
obs_out = np.bincount(co[ev], weights=y, minlength=len(pays))
obs_in = np.bincount(cd[ev], weights=y, minlength=len(pays))


def cover(draws, obs):
    lo, hi = np.percentile(draws, [2.5, 97.5], axis=0)
    return ((obs >= lo) & (obs <= hi)).mean() * 100


intervals = pd.Series({'coverage, all corridors (%)': inside.mean() * 100, 'coverage, flows > 0 (%)': inside[yb].mean() * 100,
                       'coverage, zero flows (%)': inside[~yb].mean() * 100,
                       'mean width': (R['q_hi'] - R['q_lo'])[ev].mean(),
                       'coverage, emigrants Out (%)': cover(OUT_draws, obs_out),
                       'coverage, immigrants In (%)': cover(IN_draws, obs_in),
                       'coverage, net migration (%)': cover(IN_draws - OUT_draws, obs_in - obs_out)})

pbar = np.clip(p_mean[ev], 1e-9, 1 - 1e-9)
ls_exist = np.where(yb, np.log(pbar), np.log1p(-pbar))                   # existence part of the log score
ls_vol = np.where(yb, R['lpmf'][ev], np.nan)                             # volume part, flows > 0
scores = pd.Series({'AUC': roc_auc_score(yb, pbar), 'Brier': np.mean((pbar - yb) ** 2),
                    'log score, existence': ls_exist.mean(), 'log score, volume (flows > 0)': np.nanmean(ls_vol),
                    'log score, total': (ls_exist + np.nan_to_num(ls_vol)).mean(), 'CRPS': R['crps'][ev].mean()})
t_pred = time.time() - t0

print(f"Window {T}: {ev.sum():,} corridors, {yb.sum():,} open | lambda = {LAMBDA}")
print("\nPoint forecasts (persistence: the flow of the previous window):")
display(point.round(2))
print("95% intervals (Out, In, net: totals per country, share of countries covered):")
display(intervals.round(2).to_frame('value'))
print("Scores (log scores: higher is better; Brier and CRPS: lower is better):")
display(scores.round(4).to_frame('value'))

metrics = dict(N_h=int(N_h), N_v=len(vol), D_v=len(dyads_v), N_eval=int(ev.sum()), N_eval_pos=int(yb.sum()),
               unseen_eval=int(unseen.sum()), **{k: float(v) for k, v in point.loc['median'].items()},
               **{f'{k} [{r}]': float(v) for r in point.index for k, v in point.loc[r].items()},
               **{k: float(v) for k, v in intervals.items()}, **{k: float(v) for k, v in scores.items()},
               stan_divergences=n_div, stan_rhat_max=float(p_rh.max()), stan_ess_min=float(min(p_es.min(), p_et.min())),
               **{c: float(v) for c, v, _, _ in stan_checks + bart_checks},
               **{f'BART chains: {k}': float(v) for k, v in bart_chain.items()},
               time_bart=t_bart, time_stan=t_stan, time_pred=t_pred)
out = test.loc[ev, ['orig', 'dest', 'flow', 'flow_prev', 'is_mig_lag']].assign(
    p=pbar, p_med=p_med[ev], y_median=pred['median'], y_wmedian=pred['wmedian'], y_bayes=pred['bayes'],
    q_lo=R['q_lo'][ev], q_hi=R['q_hi'][ev], unseen=unseen[ev], ls_exist=ls_exist, ls_vol=ls_vol, crps=R['crps'][ev])
out.to_csv(OUT / f'forecasts_{T}.csv.gz', index=False)
settings = dict(T=T, stan=[N_CHAINS, ITER_WARMUP, ITER_SAMPLING], bart=[BART_CHAINS, BART_GFR, BART_BURN, BART_DRAWS],
                seeds=[SEED, SEED_BART], lambda_=LAMBDA)
json.dump({'run': RUN, 'settings': settings, 'metrics': metrics, 'X_VOL': X_VOL, 'BART_VARS': BART_VARS,
           'params': params_tab.round(4).reset_index().to_dict('records'),
           'bart_importance': {k: float(v) for k, v in bart_imp.items()}}, open(OUT / 'metrics.json', 'w'), indent=1)
print(f"Saved in {OUT} (predictive: {t_pred:.0f} s)")

## Comparing runs

`paired(a, b)`: mean over corridors of the score of run $b$ minus that of run $a$, with 95% intervals from 2,000
bootstrap resamples of the origin countries, then of the destination countries (corridors of a country move together).

In [ ]:
runs = {f.parent.name: json.load(open(f))['metrics'] for f in sorted(RUNS.glob('*/metrics.json'))}
show = ['FP', 'FN', 'MAE', 'MAPE (%)', 'coverage, all corridors (%)', 'coverage, flows > 0 (%)',
        'coverage, emigrants Out (%)', 'coverage, immigrants In (%)', 'coverage, net migration (%)', 'Brier',
        'log score, existence', 'log score, volume (flows > 0)', 'log score, total', 'CRPS', 'stan_divergences',
        'stan_rhat_max']
display(pd.DataFrame({r: {c: m.get(c) for c in show} for r, m in runs.items()}).T.round(3))


def paired(a, b, n_boot=2000):
    """Run b minus run a per corridor (same window and countries). Log scores: higher is better; CRPS and absolute
    error: lower is better. 95% intervals from bootstraps over origin and over destination countries."""
    fa, fb = (pd.read_csv(next((RUNS / r).glob('forecasts_*.csv.gz'))) for r in (a, b))
    d = fa.merge(fb, on=['orig', 'dest'], suffixes=('_a', '_b'))
    assert len(d) == len(fa) == len(fb), 'runs on different corridors'
    diff = pd.DataFrame({
        'log score, total': (d['ls_exist_b'] + d['ls_vol_b'].fillna(0)) - (d['ls_exist_a'] + d['ls_vol_a'].fillna(0)),
        'log score, existence': d['ls_exist_b'] - d['ls_exist_a'],
        'log score, volume (flows > 0)': d['ls_vol_b'] - d['ls_vol_a'],
        'CRPS': d['crps_b'] - d['crps_a'],
        'absolute error, median rule': (d['flow_a'] - d['y_median_b']).abs() - (d['flow_a'] - d['y_median_a']).abs()})
    res = {'b - a': diff.mean()}
    for side in ('orig', 'dest'):
        grp = diff.groupby(d[side])
        s, n = grp.sum().values, grp.count().values                     # count skips the NaN of the volume part
        i = np.random.default_rng(0).integers(0, len(s), (n_boot, len(s)))
        lo, hi = np.percentile(s[i].sum(1) / n[i].sum(1), [2.5, 97.5], axis=0)
        res[f'95% CI, by {"origin" if side == "orig" else "destination"}'] = [f'[{l:.4g}, {h:.4g}]' for l, h in zip(lo, hi)]
    return pd.DataFrame(res, index=diff.columns)


# Example, after a second run named 'xgboost' (with 'ml_grav' in X_VOL): display(paired('main', 'xgboost'))